# Optimal transport for the LLS dataset: surface flow and the second channel under the membrane

Step 5 of the LLS pipeline (after segmentation, alignment/shape metrics and PCA). Runs the optimal transport (OT)
flow between the meshes of consecutive frames (`neutrophil_shape.CustomFunctions.optimal_transport`), tracks particles on
the surface (normal velocity, area change) and samples the second channel (**actin or myosin; the DNA movies are left
out**) along the inward normal under every particle.

Inputs: the real-space (micron) meshes in `<lls localdir>/meshes`, the `position_info` files (crop boxes, `structure`)
and the cropped raw images in `processed_images` (CZYX, channel 0 = membrane, channel 1 = second channel).
Nothing here depends on the PCA or the alignment, so it does not need `All_Data_with_CGPS_bins.csv`.

Outputs: one run folder per stretch of consecutive frames in `data/ot_lls/` (see `lls_batch.py`), summary tables and figures
in `data/ot_lls/summary` and `data/ot_lls/figures`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from neutrophil_shape.config.loader import load_config
from neutrophil_shape.CustomFunctions.optimal_transport import adapters, lls_batch

config = load_config(microscope_type='lls')
xyres, zstep, dt = adapters.get_resolution(config)
OUT_ROOT = lls_batch.default_out_root(config)   # data/ot_lls
(OUT_ROOT / 'summary').mkdir(parents=True, exist_ok=True)
(OUT_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
print(OUT_ROOT, xyres, zstep, dt)

## Which frames are run

Movies are cut into stretches of consecutive frames (OT pairs neighbouring meshes), at least 10 frames long and at most
60 frames per run so the work spreads over many processes. Only `actin` and `myosin` movies are used, `nucleus` (SPY650-DNA)
movies are excluded.

In [ ]:
table = lls_batch.load_movie_table(config)           # structure in ('actin', 'myosin')
stretches = lls_batch.find_stretches(table, max_frames=60, min_frames=10)
stretches.to_csv(OUT_ROOT / 'stretches.csv', index=False)
print(stretches.groupby('structure').agg(runs=('name', 'size'), frames=('n_frames', 'sum'), movies=('movie', 'nunique')))

## Run the OT

Each OT pair takes about 2 min on one core (the 40 frame test cell took ~80 min), so the whole dataset is hours of work on
all cores and is meant to be run from a shell, in the background, in the `nsa` environment:

```
python -m neutrophil_shape.CustomFunctions.optimal_transport.lls_batch --workers 50 --shard 0/2
python -m neutrophil_shape.CustomFunctions.optimal_transport.lls_batch --workers 50 --shard 1/2
```

It is resumable (finished runs/pairs are skipped), so it can be stopped and restarted. Set `RUN_OT = True` to run it from this
notebook instead (it blocks until everything is done).

In [ ]:
RUN_OT = False
if RUN_OT:
    if __name__ == '__main__':
        lls_batch.run_all(stretches, OUT_ROOT, workers=50)

status = lls_batch.run_status(stretches, OUT_ROOT)
status['done'] = status.sampled
print(f"{status.done.sum()} / {len(status)} runs finished, {status.pairs_done.sum()} / {status.pairs_total.sum()} pairs, "
      f"{status.error.sum()} with errors")
status.groupby('structure')[['done']].sum()

## Load the finished runs

For every particle and frame: the normal velocity and area rate of the surface (from the OT flow), and the second channel
along the inward normal (depth 0 = on the membrane, positive = under the surface, negative = outside, in voxels of `xyres`).
The second channel is expressed relative to the cell's own signal: the outside background is subtracted and the result is
divided by the median sub-membrane shell signal (depth 1-6 voxels, 0.15-0.9 um) of that frame, so cells and movies with different
brightness are comparable.

In [ ]:
rng = np.random.default_rng(0)
MAX_PER_RUN = 4000       # particle-frames kept per run for the pooled plots
runs, pooled, profiles, memprofiles = {}, [], [], []
depths = None

for _, r in status[status.done].iterrows():
    z = np.load(OUT_ROOT / r['name'] / 'ot_flow_lab' / 'actin' / 'actin_particles.npz')
    depths = z['depths']
    bg_lo = depths <= -3
    prof = z['actin_profile'] - z['actin_profile'][:, :, bg_lo].mean(2, keepdims=True)          # (T, N, D)
    mprof = z['membrane_profile'] - z['membrane_profile'][:, :, bg_lo].mean(2, keepdims=True)
    scale = np.nanmedian(z['actin_shell'], axis=1)                                              # (T,) per-frame median shell
    scale = np.where(scale > 0, scale, np.nan)
    mscale = np.nanmedian(z['membrane_shell'], axis=1)
    shell = z['actin_shell'] / scale[:, None]
    T = shell.shape[0]
    pad = lambda a: np.concatenate([a, np.full((T - a.shape[0],) + a.shape[1:], np.nan)])      # flow fields have T-1 frames
    vn = pad(z['normal_velocity']) * xyres / z['dt']                                            # um/s, > 0 protruding
    area, conf = pad(z['log2_area_rate']), pad(z['confidence'])
    runs[r['name']] = dict(structure=r['structure'], shell=shell, vn=vn, area=area,
                           conf=conf, prof=prof / scale[:, None, None], dt=float(z['dt']))
    ok = np.isfinite(shell) & np.isfinite(vn) & (conf >= 0.5)
    ti, pi = np.nonzero(ok)
    keep = rng.choice(len(ti), size=min(len(ti), MAX_PER_RUN), replace=False)
    ti, pi = ti[keep], pi[keep]
    pooled.append(pd.DataFrame(dict(run=r['name'], structure=r['structure'], T=ti, particle=pi,
                                    shell=shell[ti, pi], vn_um_s=vn[ti, pi], log2_area_rate=area[ti, pi])))
    profiles.append((prof / scale[:, None, None])[ti, pi])
    memprofiles.append((mprof / mscale[:, None, None])[ti, pi])

pooled = pd.concat(pooled, ignore_index=True) if pooled else pd.DataFrame()
profiles = np.concatenate(profiles) if profiles else np.zeros((0, 0))
memprofiles = np.concatenate(memprofiles) if memprofiles else np.zeros((0, 0))
depth_um = None if depths is None else depths * xyres
print(len(runs), 'runs,', len(pooled), 'particle-frames sampled')
pooled.groupby('structure').size() if len(pooled) else None

## Profile of the second channel under the surface

Mean over runs of the (normalized) second channel along the inward normal, for actin and myosin, with the membrane
channel for reference (the membrane peaks at depth 0). Shaded: s.e.m. over runs.

In [ ]:
COLORS = {'actin': '#1b9e77', 'myosin': '#d95f02'}

def run_means(structure, arr):
    # mean over the sampled particle-frames of every run, returns (runs, depths)
    sub = pooled.structure.values == structure
    out = [arr[sub & (pooled.run.values == rn)].mean(0) for rn in pooled.run[sub].unique()]
    return np.array(out)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
for s in ('actin', 'myosin'):
    for ax, arr, title in zip(axes, (profiles, memprofiles), ('second channel (actin / myosin)', 'membrane (CAAX)')):
        m = run_means(s, arr)
        if len(m) == 0:
            continue
        mu, sem = m.mean(0), m.std(0, ddof=1) / np.sqrt(len(m)) if len(m) > 1 else np.zeros(m.shape[1])
        ax.plot(depth_um, mu, color=COLORS[s], label=f'{s} (n={len(m)} runs)')
        ax.fill_between(depth_um, mu - sem, mu + sem, color=COLORS[s], alpha=0.25)
        ax.set_title(title)
for ax in axes:
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel('depth below the surface (um)  [<0 outside]')
axes[0].set_ylabel('signal / median shell signal')
axes[0].legend()
fig.tight_layout()
fig.savefig(OUT_ROOT / 'figures' / 'depth_profiles.png', dpi=200)

## Profile in protruding vs retracting regions

Particle-frames in the top / bottom 20% of the normal velocity (within each structure) vs the middle 60%.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=False)
for ax, s in zip(axes, ('actin', 'myosin')):
    sub = pooled.structure.values == s
    if sub.sum() == 0:
        continue
    lo, hi = np.quantile(pooled.vn_um_s[sub], [0.2, 0.8])
    classes = {'retracting (bottom 20%)': pooled.vn_um_s.values <= lo,
               'middle 60%': (pooled.vn_um_s.values > lo) & (pooled.vn_um_s.values < hi),
               'protruding (top 20%)': pooled.vn_um_s.values >= hi}
    for (name, sel), c in zip(classes.items(), ('tab:blue', 'gray', 'tab:red')):
        m = np.array([profiles[sub & sel & (pooled.run.values == rn)].mean(0)
                      for rn in pooled.run[sub].unique() if (sub & sel & (pooled.run.values == rn)).any()])
        mu, sem = m.mean(0), m.std(0, ddof=1) / np.sqrt(len(m))
        ax.plot(depth_um, mu, color=c, label=name)
        ax.fill_between(depth_um, mu - sem, mu + sem, color=c, alpha=0.2)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_title(s); ax.set_xlabel('depth below the surface (um)')
axes[0].set_ylabel('signal / median shell signal'); axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_ROOT / 'figures' / 'depth_profiles_by_velocity.png', dpi=200)

## Sub-membrane signal vs surface velocity

Normalized shell signal (1-6 voxels under the surface) binned by normal velocity (20 equal-count bins within each structure);
median with the interquartile range of the run medians.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, s in zip(axes, ('actin', 'myosin')):
    d = pooled[pooled.structure == s]
    if len(d) == 0:
        continue
    d = d.assign(bin=pd.qcut(d.vn_um_s, 20, labels=False, duplicates='drop'))
    per_run = d.groupby(['bin', 'run']).agg(vn=('vn_um_s', 'median'), shell=('shell', 'median')).reset_index()
    g = per_run.groupby('bin')
    ax.errorbar(g.vn.median(), g.shell.median(), yerr=[g.shell.median() - g.shell.quantile(.25), g.shell.quantile(.75) - g.shell.median()],
                fmt='o-', color=COLORS[s], ms=3, lw=1)
    ax.axvline(0, color='k', lw=0.5); ax.axhline(1, color='k', lw=0.5, ls=':')
    rho = stats.spearmanr(d.vn_um_s, d.shell)
    ax.set_title(f'{s}: Spearman rho = {rho.statistic:.2f} (pooled)')
    ax.set_xlabel('normal velocity (um/s, >0 protruding)')
axes[0].set_ylabel('sub-membrane signal / frame median')
fig.tight_layout()
fig.savefig(OUT_ROOT / 'figures' / 'shell_vs_normal_velocity.png', dpi=200)

## Lagged correlation along the particle trajectories

Spearman correlation between the sub-membrane signal at frame t and the normal velocity at frame t+lag of the same
particle (lag > 0: the signal comes before the velocity). One value per run, then the mean and s.e.m. over runs per structure.

In [ ]:
LAGS = np.arange(-8, 9)
rows = []
for name, r in runs.items():
    T = r['shell'].shape[0]
    for lag in LAGS:
        if abs(lag) >= T:
            continue
        if lag >= 0:
            a, v = r['shell'][:T - lag], r['vn'][lag:]
        else:
            a, v = r['shell'][-lag:], r['vn'][:T + lag]
        ok = np.isfinite(a) & np.isfinite(v)
        if ok.sum() > 200:
            rows.append(dict(run=name, structure=r['structure'], lag=lag, rho=stats.spearmanr(a[ok], v[ok]).statistic))
lagdf = pd.DataFrame(rows)
if len(lagdf):
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    for s in ('actin', 'myosin'):
        g = lagdf[lagdf.structure == s].groupby('lag').rho
        if len(g) == 0:
            continue
        mu, sem = g.mean(), g.std() / np.sqrt(g.count())
        ax.plot(mu.index * dt, mu, color=COLORS[s], label=f'{s} (n={lagdf[lagdf.structure == s].run.nunique()})')
        ax.fill_between(mu.index * dt, mu - sem, mu + sem, color=COLORS[s], alpha=0.25)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel('lag of normal velocity after signal (s)'); ax.set_ylabel('Spearman rho')
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT_ROOT / 'figures' / 'lagged_correlation.png', dpi=200)

## Per-run summary and an example run for each structure

In [ ]:
summ = []
for name, r in runs.items():
    ok = np.isfinite(r['shell']) & np.isfinite(r['vn'])
    summ.append(dict(run=name, structure=r['structure'], frames=r['shell'].shape[0], particles=r['shell'].shape[1],
                     median_confidence=np.nanmedian(r['conf']),
                     median_abs_normal_velocity_um_s=np.nanmedian(np.abs(r['vn'])),
                     rho_shell_vs_velocity=stats.spearmanr(r['shell'][ok], r['vn'][ok]).statistic if ok.sum() > 200 else np.nan))
summ = pd.DataFrame(summ)
summ.to_csv(OUT_ROOT / 'summary' / 'run_summary.csv', index=False)
display_cols = summ.groupby('structure')[['particles', 'median_confidence', 'rho_shell_vs_velocity']].median()
display_cols

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
for row, s in enumerate(('actin', 'myosin')):
    cand = summ[summ.structure == s]
    if len(cand) == 0:
        continue
    name = cand.sort_values('frames').run.iloc[-1]
    r = runs[name]
    alive = np.isfinite(r['shell']).sum(0) >= r['shell'].shape[0] * 0.9           # particles present in most frames
    order = np.argsort(np.nanmean(r['shell'][:, alive], axis=0))
    t = np.arange(r['shell'].shape[0]) * r['dt']
    for ax, arr, lab, cmap, lim in ((axes[row, 0], r['shell'], 'signal / frame median', 'magma', (0, 3)),
                                    (axes[row, 1], r['vn'], 'normal velocity (um/s)', 'coolwarm', None)):
        a = arr[:, alive][:, order].T
        lim = lim or (-np.nanpercentile(np.abs(a), 95), np.nanpercentile(np.abs(a), 95))
        im = ax.imshow(a, aspect='auto', cmap=cmap, vmin=lim[0], vmax=lim[1], extent=[t[0], t[-1], 0, a.shape[0]])
        fig.colorbar(im, ax=ax, label=lab)
        ax.set_xlabel('time (s)'); ax.set_ylabel('particles (ordered by mean signal)')
    axes[row, 0].set_title(f'{s}: {name[:60]}...', fontsize=8)
fig.tight_layout()
fig.savefig(OUT_ROOT / 'figures' / 'example_runs.png', dpi=200)